In [1]:
import sys
import os
import numpy as np
import pandas as pd
import time

from sklearn.ensemble import RandomForestClassifier

sys.path.append(os.path.join(os.getcwd(), '..', '..'))
from src.preprocessing import apply_smote
from src.evaluation import compute_metrics, save_results

RANDOM_STATE = 42

In [2]:

DATA_DIR = "../../data/processed"

X_train_full = pd.read_pickle(f"{DATA_DIR}/X_train_scaled.pkl")
X_test_full = pd.read_pickle(f"{DATA_DIR}/X_test_scaled.pkl")
y_train = pd.read_pickle(f"{DATA_DIR}/y_train.pkl")
y_test = pd.read_pickle(f"{DATA_DIR}/y_test.pkl")
target_encoder = pd.read_pickle(f"{DATA_DIR}/target_encoder.pkl")
target_names = list(target_encoder.classes_)

if isinstance(y_train, pd.Series):
    y_train = y_train.values
if isinstance(y_test, pd.Series):
    y_test = y_test.values

X_train_top10 = pd.read_pickle(f"{DATA_DIR}/X_train_top10.pkl")
X_test_top10 = pd.read_pickle(f"{DATA_DIR}/X_test_top10.pkl")
X_train_top20 = pd.read_pickle(f"{DATA_DIR}/X_train_top20.pkl")
X_test_top20 = pd.read_pickle(f"{DATA_DIR}/X_test_top20.pkl")
X_train_top30 = pd.read_pickle(f"{DATA_DIR}/X_train_top30.pkl")
X_test_top30 = pd.read_pickle(f"{DATA_DIR}/X_test_top30.pkl")
X_train_pca = pd.read_pickle(f"{DATA_DIR}/X_train_pca.pkl")
X_test_pca = pd.read_pickle(f"{DATA_DIR}/X_test_pca.pkl")

print(f"Data loaded. Classes: {len(target_names)}")
print(f"Feature sets: full={X_train_full.shape[1]}, top10={X_train_top10.shape[1]}, "
      f"top20={X_train_top20.shape[1]}, top30={X_train_top30.shape[1]}, pca={X_train_pca.shape[1]}")


Data loaded. Classes: 12
Feature sets: full=83, top10=10, top20=20, top30=30, pca=26


In [5]:

def run_rf_experiment(X_train, y_train, X_test, y_test,
                      feature_set_name, imbalance_strategy,
                      target_encoder=None):
    """
    Run one Random Forest experiment.
    """
    model_name = f"RF | {feature_set_name} | {imbalance_strategy}"

    print(f"# {model_name}")


    # Apply imbalance strategy
    if imbalance_strategy == "smote":
        print("  Applying SMOTE...")
        X_tr, y_tr = apply_smote(X_train, y_train)
    else:
        X_tr, y_tr = X_train, y_train

    # Create model
    if imbalance_strategy == "cost-sensitive":
        model = RandomForestClassifier(
            n_estimators=100,
            class_weight='balanced',
            random_state=RANDOM_STATE,
            n_jobs=-1  # use all CPU cores for speed
        )
    else:
        model = RandomForestClassifier(
            n_estimators=100,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )

    # Train
    start_time = time.time()
    print("  Training...")
    model.fit(X_tr, y_tr)
    train_time = time.time() - start_time
    print(f"  Training time: {train_time:.1f}s")

    # Predict
    y_pred = model.predict(X_test)

    # Evaluate
    metrics = compute_metrics(y_test, y_pred, target_encoder=target_encoder)
    metrics["feature_set"] = feature_set_name
    metrics["imbalance_strategy"] = imbalance_strategy
    metrics["training_time"] = train_time

    # Save results
    exp_name = f"rf_{feature_set_name}_{imbalance_strategy}"
    save_results(metrics, exp_name, save_dir="../../results")

    return metrics


In [6]:

feature_sets = {
    "full-83": (X_train_full, X_test_full),
    "top-30": (X_train_top30, X_test_top30),
    "top-20": (X_train_top20, X_test_top20),
    "top-10": (X_train_top10, X_test_top10),
    "pca": (X_train_pca, X_test_pca),
}

imbalance_strategies = ["baseline", "smote", "cost-sensitive"]

all_results = {}

for fs_name, (X_tr, X_te) in feature_sets.items():
    for strategy in imbalance_strategies:
        exp_name = f"RF | {fs_name} | {strategy}"
        result = run_rf_experiment(
            X_train=X_tr,
            y_train=y_train,
            X_test=X_te,
            y_test=y_test,
            feature_set_name=fs_name,
            imbalance_strategy=strategy,
            target_encoder=target_encoder,
        )
        all_results[exp_name] = result

print("\n\nAll 15 RF experiments complete!")

# RF | full-83 | baseline
  Training...
  Training time: 0.5s

  Macro F1 (PRIMARY):  0.9763
  Accuracy:            0.9985
  Macro Precision:     0.9776
  Macro Recall:        0.9788

  Saved report to ../../results/rf_full-83_baseline_report.csv
  Saved summary to ../../results/rf_full-83_baseline_summary.csv
# RF | full-83 | smote
  Applying SMOTE...
  Before SMOTE: 98,493 samples


/Users/mahip/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


  After SMOTE:  908,724 samples
  Training...
  Training time: 20.7s

  Macro F1 (PRIMARY):  0.9668
  Accuracy:            0.9988
  Macro Precision:     0.9544
  Macro Recall:        0.9823

  Saved report to ../../results/rf_full-83_smote_report.csv
  Saved summary to ../../results/rf_full-83_smote_summary.csv
# RF | full-83 | cost-sensitive
  Training...
  Training time: 0.6s

  Macro F1 (PRIMARY):  0.9652
  Accuracy:            0.9985
  Macro Precision:     0.9538
  Macro Recall:        0.9798

  Saved report to ../../results/rf_full-83_cost-sensitive_report.csv
  Saved summary to ../../results/rf_full-83_cost-sensitive_summary.csv
# RF | top-30 | baseline
  Training...
  Training time: 0.3s

  Macro F1 (PRIMARY):  0.9658
  Accuracy:            0.9986
  Macro Precision:     0.9543
  Macro Recall:        0.9805

  Saved report to ../../results/rf_top-30_baseline_report.csv
  Saved summary to ../../results/rf_top-30_baseline_summary.csv
# RF | top-30 | smote
  Applying SMOTE...
  Befo

/Users/mahip/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


  After SMOTE:  908,724 samples
  Training...
  Training time: 11.4s

  Macro F1 (PRIMARY):  0.9665
  Accuracy:            0.9986
  Macro Precision:     0.9541
  Macro Recall:        0.9820

  Saved report to ../../results/rf_top-30_smote_report.csv
  Saved summary to ../../results/rf_top-30_smote_summary.csv
# RF | top-30 | cost-sensitive
  Training...
  Training time: 0.4s

  Macro F1 (PRIMARY):  0.9697
  Accuracy:            0.9985
  Macro Precision:     0.9638
  Macro Recall:        0.9781

  Saved report to ../../results/rf_top-30_cost-sensitive_report.csv
  Saved summary to ../../results/rf_top-30_cost-sensitive_summary.csv
# RF | top-20 | baseline
  Training...
  Training time: 0.3s

  Macro F1 (PRIMARY):  0.9593
  Accuracy:            0.9976
  Macro Precision:     0.9480
  Macro Recall:        0.9736

  Saved report to ../../results/rf_top-20_baseline_report.csv
  Saved summary to ../../results/rf_top-20_baseline_summary.csv
# RF | top-20 | smote
  Applying SMOTE...
  Before SM

...

  After SMOTE:  908,724 samples
  Training...
  Training time: 8.0s

  Macro F1 (PRIMARY):  0.9607
  Accuracy:            0.9972
  Macro Precision:     0.9515
  Macro Recall:        0.9714

  Saved report to ../../results/rf_top-20_smote_report.csv
  Saved summary to ../../results/rf_top-20_smote_summary.csv
# RF | top-20 | cost-sensitive
  Training...
  Training time: 0.3s

  Macro F1 (PRIMARY):  0.9532
  Accuracy:            0.9975
  Macro Precision:     0.9391
  Macro Recall:        0.9721

  Saved report to ../../results/rf_top-20_cost-sensitive_report.csv
  Saved summary to ../../results/rf_top-20_cost-sensitive_summary.csv
# RF | top-10 | baseline
  Training...
  Training time: 0.2s

  Macro F1 (PRIMARY):  0.9526
  Accuracy:            0.9972
  Macro Precision:     0.9419
  Macro Recall:        0.9688

  Saved report to ../../results/rf_top-10_baseline_report.csv
  Saved summary to ../../results/rf_top-10_baseline_summary.csv
# RF | top-10 | smote
  Applying SMOTE...
  Before SMO

/Users/mahip/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


  Training time: 5.9s

  Macro F1 (PRIMARY):  0.9574
  Accuracy:            0.9972
  Macro Precision:     0.9458
  Macro Recall:        0.9737

  Saved report to ../../results/rf_top-10_smote_report.csv
  Saved summary to ../../results/rf_top-10_smote_summary.csv
# RF | top-10 | cost-sensitive
  Training...
  Training time: 0.2s

  Macro F1 (PRIMARY):  0.9435
  Accuracy:            0.9971
  Macro Precision:     0.9229
  Macro Recall:        0.9734

  Saved report to ../../results/rf_top-10_cost-sensitive_report.csv
  Saved summary to ../../results/rf_top-10_cost-sensitive_summary.csv
# RF | pca | baseline
  Training...
  Training time: 2.8s

  Macro F1 (PRIMARY):  0.9708
  Accuracy:            0.9978
  Macro Precision:     0.9784
  Macro Recall:        0.9682

  Saved report to ../../results/rf_pca_baseline_report.csv
  Saved summary to ../../results/rf_pca_baseline_summary.csv
# RF | pca | smote
  Applying SMOTE...
  Before SMOTE: 98,493 samples


/Users/mahip/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


  After SMOTE:  908,724 samples
  Training...
  Training time: 47.1s

  Macro F1 (PRIMARY):  0.9708
  Accuracy:            0.9979
  Macro Precision:     0.9731
  Macro Recall:        0.9723

  Saved report to ../../results/rf_pca_smote_report.csv
  Saved summary to ../../results/rf_pca_smote_summary.csv
# RF | pca | cost-sensitive
  Training...
  Training time: 4.0s

  Macro F1 (PRIMARY):  0.9673
  Accuracy:            0.9976
  Macro Precision:     0.9736
  Macro Recall:        0.9656

  Saved report to ../../results/rf_pca_cost-sensitive_report.csv
  Saved summary to ../../results/rf_pca_cost-sensitive_summary.csv


All 15 RF experiments complete!


In [7]:
from src.evaluation import compare_experiments
compare_experiments(all_results)


  EXPERIMENT COMPARISON (sorted by Macro F1)
                               Macro F1  Accuracy  Macro Precision  Macro Recall
Experiment                                                                      
RF | full-83 | baseline          0.9763    0.9985           0.9776        0.9788
RF | pca | baseline              0.9708    0.9978           0.9784        0.9682
RF | pca | smote                 0.9708    0.9979           0.9731        0.9723
RF | top-30 | cost-sensitive     0.9697    0.9985           0.9638        0.9781
RF | pca | cost-sensitive        0.9673    0.9976           0.9736        0.9656
RF | full-83 | smote             0.9668    0.9988           0.9544        0.9823
RF | top-30 | smote              0.9665    0.9986           0.9541        0.9820
RF | top-30 | baseline           0.9658    0.9986           0.9543        0.9805
RF | full-83 | cost-sensitive    0.9652    0.9985           0.9538        0.9798
RF | top-20 | smote              0.9607    0.9972           0.9

,Macro F1,Accuracy,Macro Precision,Macro Recall
Experiment,,,,
RF | full-83 | baseline,0.976337,0.998497,0.977567,0.978751
RF | pca | baseline,0.970819,0.997807,0.978439,0.968209
RF | pca | smote,0.970784,0.997888,0.973081,0.972325
RF | top-30 | cost-sensitive,0.969737,0.998538,0.963783,0.978075
RF | pca | cost-sensitive,0.967254,0.997604,0.973565,0.965593
RF | full-83 | smote,0.966825,0.998822,0.954406,0.982323
RF | top-30 | smote,0.966509,0.998579,0.954080,0.982019
RF | top-30 | baseline,0.965832,0.998619,0.954271,0.980490
RF | full-83 | cost-sensitive,0.965219,0.998538,0.953766,0.979783


In [8]:
results_dir = "../../results"
for f in sorted(os.listdir(results_dir)):
    if f.startswith("rf_") and f.endswith("_report.csv"):
        name = f.replace("_report.csv", "")
        df = pd.read_csv(os.path.join(results_dir, f), index_col=0)
        metasploit = df.loc["Metasploit_Brute_Force_SSH"] if "Metasploit_Brute_Force_SSH" in df.index else df.iloc[4]
        nmap_fin = df.loc["NMAP_FIN_SCAN"] if "NMAP_FIN_SCAN" in df.index else df.iloc[5]
        print(f"\n{name}")
        print(f"  Metasploit_Brute_Force_SSH: precision={metasploit['precision']:.3f}  recall={metasploit['recall']:.3f}  f1={metasploit['f1-score']:.3f}")
        print(f"  NMAP_FIN_SCAN:              precision={nmap_fin['precision']:.3f}  recall={nmap_fin['recall']:.3f}  f1={nmap_fin['f1-score']:.3f}")

# %%


rf_full-83_baseline
  Metasploit_Brute_Force_SSH: precision=0.778  recall=1.000  f1=0.875
  NMAP_FIN_SCAN:              precision=1.000  recall=0.833  f1=0.909

rf_full-83_cost-sensitive
  Metasploit_Brute_Force_SSH: precision=0.778  recall=1.000  f1=0.875
  NMAP_FIN_SCAN:              precision=0.714  recall=0.833  f1=0.769

rf_full-83_smote
  Metasploit_Brute_Force_SSH: precision=0.778  recall=1.000  f1=0.875
  NMAP_FIN_SCAN:              precision=0.714  recall=0.833  f1=0.769

rf_pca_baseline
  Metasploit_Brute_Force_SSH: precision=0.778  recall=1.000  f1=0.875
  NMAP_FIN_SCAN:              precision=1.000  recall=0.833  f1=0.909

rf_pca_cost-sensitive
  Metasploit_Brute_Force_SSH: precision=0.778  recall=1.000  f1=0.875
  NMAP_FIN_SCAN:              precision=1.000  recall=0.833  f1=0.909

rf_pca_smote
  Metasploit_Brute_Force_SSH: precision=0.778  recall=1.000  f1=0.875
  NMAP_FIN_SCAN:              precision=1.000  recall=0.833  f1=0.909

rf_top-10_baseline
  Metasploit_Brute_F

In [11]:
print("\n" + "-"*20)
print("  CROSS-MODEL COMPARISON — Best from each model")
print("-"*20)

summaries = []
for f in sorted(os.listdir(results_dir)):
    if f.endswith("_summary.csv"):
        df = pd.read_csv(os.path.join(results_dir, f))
        summaries.append(df)

all_df['model_type'] = all_df['experiment'].apply(lambda x: x.split('_')[0])
best_per_model = all_df.loc[all_df.groupby('model_type')['macro_f1'].idxmax()]
print(best_per_model.sort_values('macro_f1', ascending=False).to_string(index=False))
print("-"*20)


--------------------
  CROSS-MODEL COMPARISON — Best from each model
--------------------
         experiment  macro_f1  accuracy  macro_precision  macro_recall model_type
rf_full-83_baseline  0.976337  0.998497         0.977567      0.978751         rf
dnn_top-20_baseline  0.951497  0.994558         0.970280      0.936111        dnn
lr_full-83_baseline  0.917922  0.991066         0.935213      0.908956         lr
--------------------
